# Milestone 1: Data Audit Notebook
**Renewable Energy-Aware Load Prediction Model for Smart Grids**  
*Pavagada Solar Park (2,050 MW), Tumkur District, Karnataka*

This notebook performs an initial audit on `data/raw/final_dataset.csv`.
In accordance with our thin-notebook architecture, all audit and validation logic is housed in `src/audit.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Set up project root path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import config
from src.audit import run_data_audit, generate_audit_report

# Execute data audit harness
results = run_data_audit(config.RAW_DATA_FILE)
print(f"Total Rows: {results['total_rows']:,}")
print(f"Total Columns: {results['total_cols']}")
print(f"Earliest Timestamp: {results['dt_min']}")
print(f"Latest Timestamp: {results['dt_max']}")
print(f"Duplicate Timestamps: {results['num_duplicates']}")
print(f"Missing Hours in Range: {results['missing_hours']}")

Total Rows: 43,824
Total Columns: 69
Earliest Timestamp: 2021-03-01 00:00:00
Latest Timestamp: 2026-02-28 23:00:00
Duplicate Timestamps: 0
Missing Hours in Range: 0


### Temporal Continuity Analysis
The dataset spans exactly 5 years (2021-03-01 00:00 to 2026-02-28 23:00 IST), containing **43,824 hourly observations** with **zero duplicates** and **zero missing hours**.

In [2]:
# Missing values breakdown
print("Missing Value Breakdown:")
results['missing_summary']

Missing Value Breakdown:


,missing_count,missing_pct
ghi_wm2,158,0.360533
dni_wm2,158,0.360533
dhi_wm2,158,0.360533
poa_irradiance_wm2,158,0.360533
temperature_2m_c,177,0.403888
module_temp_c,158,0.360533
relative_humidity_pct,177,0.403888
wind_speed_10m_ms,177,0.403888
wind_direction_deg,177,0.403888
cloud_cover_total_pct,177,0.403888


### Missingness Findings
- `solar_power_mw`: 0 missing records (complete physics-based calculation).
- Weather columns (`ghi_wm2`, `dni_wm2`, `temperature_2m_c`, etc.): 0.36% to 0.40% missing (158 to 177 rows).
- `demand_mw`: 0.349% missing (153 rows).
All gaps are sparse and short-lived, appropriate for linear interpolation in Milestone 2.

In [3]:
# Physical sanity verification
print(f"Max night solar (elevation <= 0): {results['night_solar_max']:.4f} MW")
print(f"Capacity factor bounds violations: {results['cf_violations']}")
print(f"Relative humidity bounds violations: {results['rh_violations']}")
print(f"Max absolute error in section summation: {results['max_section_diff']:.6f} MW")
print(f"Max absolute error in net load identity: {results['max_net_diff']:.6f} MW")
print(f"Outage/Curtailment hours: {results['outage_count']} ({results['outage_pct']:.2f}%)")
print(f"Outlier hours: {results['outlier_count']} ({results['outlier_pct']:.2f}%)")

Max night solar (elevation <= 0): 0.0000 MW
Capacity factor bounds violations: 0
Relative humidity bounds violations: 0
Max absolute error in section summation: 0.003000 MW
Max absolute error in net load identity: 0.010000 MW
Outage/Curtailment hours: 280 (0.64%)
Outlier hours: 78 (0.18%)


### Physical and Engineering Sanity
1. **Night Solar**: Solar generation is strictly 0.0 MW whenever solar elevation $\le$ 0°.
2. **Capacity Factor**: Values remain firmly bounded within $[0.0, 1.0]$.
3. **Section Additivity**: Sum of sections 1 through 8 matches `solar_power_mw` within numerical precision.
4. **Net Load**: Strictly satisfies $P_{\text{net}} = P_{\text{demand}} - P_{\text{solar}}$.
5. **Outages & Outliers**: 280 outage hours (0.64%) and 78 outlier hours (0.18%) exist. As noted in Section 2A, these flags will be excluded from the feature matrix to avoid lookahead leakage, but tracked for evaluation.

In [4]:
# Yearly demand progression
yearly_demand = pd.DataFrame(list(results['yearly_demand'].items()), columns=['Year', 'Mean_Demand_MW'])
yearly_demand

,Year,Mean_Demand_MW
0,2021,10165.176956
1,2022,10751.430794
2,2023,11347.567017
3,2024,11979.050795
4,2025,12593.617206
5,2026,13487.387334


### Trend Observations
Mean demand escalates from 10,165 MW in 2021 to 12,594 MW in 2025 (~5.5% annual growth) and 13,487 MW in Jan-Feb 2026. This upward shift requires models (especially tree regressors) to rely on relative lags and rolling statistics rather than raw calendar year.

In [5]:
# Descriptive statistics
results['df_describe'].round(2)

,solar_power_mw,demand_mw,net_load_mw,ghi_wm2,temperature_2m_c,relative_humidity_pct
count,43824.00,43671.00,43671.00,43666.00,43647.00,43647.00
mean,443.34,11475.00,11031.50,230.72,26.57,59.33
std,584.70,2147.87,2001.89,306.96,4.88,15.14
min,0.00,6200.00,6200.00,0.00,14.00,15.83
25%,0.00,9914.02,9556.12,0.00,23.24,47.77
50%,1.30,11367.54,10873.12,0.63,26.42,60.91
75%,882.84,12955.31,12351.00,455.41,29.45,71.19
max,1875.75,18534.54,18534.54,1136.26,43.50,99.00


## Conclusion & Milestone 1 Checkpoint Summary
The data audit is complete. All columns and date ranges match `src/config.py` specifications.
Audit results and markdown summaries have been written to `results/tables/data_audit.md`.
The pipeline is ready for Milestone 2: Preprocessing.